<a href="https://colab.research.google.com/github/misbahhassan6400/flyrank-ml-internship/blob/main/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

One row in this feature frame represents one client-content pair. The five features summarize observed data from February 2026. The label is March 2026 GA4 sessions, which is kept separate from the feature columns because it is only known after the February decision moment.

In [ ]:
!pip -q install duckdb huggingface_hub pandas

import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("HF_TOKEN secret missing")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

files = con.sql("""
SELECT file
FROM glob('hf://datasets/FlyRank/internship-warehouse/**/*.parquet')
""").df()

feb_matches = files[files["file"].astype(str).str.contains("2026-02", regex=False)]
mar_matches = files[files["file"].astype(str).str.contains("2026-03", regex=False)]

FEB = feb_matches.iloc[0]["file"]
MAR = mar_matches.iloc[0]["file"]

print("Setup complete")
print("FEB:", FEB)
print("MAR:", MAR)

Setup complete
FEB: hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/data_0.parquet
MAR: hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet


In [ ]:
feature_frame = con.sql(f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions)
            FILTER (WHERE gsc_data_available IS TRUE)
            AS feb_gsc_impressions,

        SUM(gsc_clicks)
            FILTER (WHERE gsc_data_available IS TRUE)
            AS feb_gsc_clicks,

        AVG(gsc_avg_position)
            FILTER (
                WHERE gsc_data_available IS TRUE
                AND gsc_avg_position IS NOT NULL
            )
            AS feb_gsc_avg_position,

        SUM(ga4_sessions)
            FILTER (WHERE ga4_data_available IS TRUE)
            AS feb_ga4_sessions,

        SUM(ga4_engaged_sessions)
            FILTER (WHERE ga4_data_available IS TRUE)
            AS feb_ga4_engaged_sessions

    FROM read_parquet('{FEB}')
    WHERE month = '2026-02'
    GROUP BY client_hash_id, content_hash_id
),

mar AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(ga4_sessions)
            FILTER (WHERE ga4_data_available IS TRUE)
            AS march_ga4_sessions

    FROM read_parquet('{MAR}')
    WHERE month = '2026-03'
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    feb.*,
    mar.march_ga4_sessions
FROM feb
INNER JOIN mar
    USING (client_hash_id, content_hash_id)
WHERE mar.march_ga4_sessions IS NOT NULL
""").df()

feature_cols = [
    "feb_gsc_impressions",
    "feb_gsc_clicks",
    "feb_gsc_avg_position",
    "feb_ga4_sessions",
    "feb_ga4_engaged_sessions",
]

print("Feature frame shape:", feature_frame.shape)
display(feature_frame.head())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (80877, 8)


,client_hash_id,content_hash_id,feb_gsc_impressions,feb_gsc_clicks,feb_gsc_avg_position,feb_ga4_sessions,feb_ga4_engaged_sessions,march_ga4_sessions
0,client_e547b89c05043229,content_9abd8b303f805847,733.0,6.0,6.495085,6.0,0.0,3.0
1,client_e547b89c05043229,content_5f58c55cbfee172a,514.0,0.0,10.490023,1.0,0.0,1.0
2,client_e547b89c05043229,content_6fe390ba3af1e456,2931.0,3.0,38.436254,6.0,1.0,4.0
3,client_e547b89c05043229,content_3ad5d2160242b9ca,970.0,2.0,9.710810,3.0,0.0,1.0
4,client_e547b89c05043229,content_a2bd730a7cf68316,551.0,1.0,6.017373,3.0,0.0,1.0


In [ ]:
missing_before = feature_frame[feature_cols].isna().sum()

for col in feature_cols:
    feature_frame[col] = feature_frame[col].fillna(
        feature_frame[col].median()
    )

feature_frame[feature_cols] = feature_frame[feature_cols].fillna(0)

print("Missing values before filling:")
display(missing_before)

print("Missing values after filling:")
display(feature_frame[feature_cols].isna().sum())

print("Feature columns:")
print(feature_cols)

Missing values before filling:


,0
feb_gsc_impressions,22617
feb_gsc_clicks,22617
feb_gsc_avg_position,22617
feb_ga4_sessions,55210
feb_ga4_engaged_sessions,55210


Missing values after filling:


,0
feb_gsc_impressions,0
feb_gsc_clicks,0
feb_gsc_avg_position,0
feb_ga4_sessions,0
feb_ga4_engaged_sessions,0


Feature columns:
['feb_gsc_impressions', 'feb_gsc_clicks', 'feb_gsc_avg_position', 'feb_ga4_sessions', 'feb_ga4_engaged_sessions']


## 2. Feature notes (meaning, missing, categorical, available-when?)

- `feb_gsc_impressions`: February ke Google Search impressions; February ke end tak knowable because daily Search Console data already arrived.
- `feb_gsc_clicks`: February ke Google Search clicks; decision moment se pehle available.
- `feb_gsc_avg_position`: February ki observed average search position; decision moment se pehle available.
- `feb_ga4_sessions`: February ki GA4 sessions; decision moment se pehle available.
- `feb_ga4_engaged_sessions`: February ki engaged GA4 sessions; decision moment se pehle available.

Missing numeric feature values are filled with the feature median calculated from the February feature frame. Remaining all-missing values are filled with zero. The March outcome is not used in the honest feature matrix

## 3. The leakage hunt

First, I score the five February features honestly. Then I deliberately add a copy of the March label as a feature. The leaky score should jump close to 1.0 because the model is being given the answer. I then remove that column and keep the honest score.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer

X_honest = feature_frame[feature_cols].copy()
y = feature_frame["march_ga4_sessions"].astype(float)

# Use the same train/test rows for both experiments
row_indices = np.arange(len(feature_frame))

train_idx, test_idx = train_test_split(
    row_indices,
    test_size=0.20,
    random_state=42
)

honest_model = make_pipeline(
    SimpleImputer(strategy="median"),
    LinearRegression()
)

honest_model.fit(
    X_honest.iloc[train_idx],
    y.iloc[train_idx]
)

honest_r2 = honest_model.score(
    X_honest.iloc[test_idx],
    y.iloc[test_idx]
)

print("Honest feature count:", len(feature_cols))
print("Honest R2:", round(honest_r2, 4))


Honest feature count: 5
Honest R2: 0.2715


In [ ]:
# Deliberate leakage: add the future label as a feature on purpose
X_leaky = X_honest.copy()
X_leaky["leaky_march_ga4_sessions"] = y.values

leaky_model = make_pipeline(
    SimpleImputer(strategy="median"),
    LinearRegression()
)

leaky_model.fit(
    X_leaky.iloc[train_idx],
    y.iloc[train_idx]
)

leaky_r2 = leaky_model.score(
    X_leaky.iloc[test_idx],
    y.iloc[test_idx]
)

print("Leaky feature count:", len(X_leaky.columns))
print("Leaky R2:", round(leaky_r2, 4))

Leaky feature count: 6
Leaky R2: 1.0


In [ ]:
X_final = X_leaky.drop(columns=["leaky_march_ga4_sessions"])

final_model = make_pipeline(
    SimpleImputer(strategy="median"),
    LinearRegression()
)

final_model.fit(
    X_final.iloc[train_idx],
    y.iloc[train_idx]
)

final_honest_r2 = final_model.score(
    X_final.iloc[test_idx],
    y.iloc[test_idx]
)

print("Leaky column removed:", "leaky_march_ga4_sessions" not in X_final.columns)
print("Final feature count:", len(X_final.columns))
print("Final honest R2:", round(final_honest_r2, 4))

assert "leaky_march_ga4_sessions" not in X_final.columns
assert len(X_final.columns) == 5

Leaky column removed: True
Final feature count: 5
Final honest R2: 0.2715


## 4. What I excluded and why

- `march_ga4_sessions`: excluded from the final feature matrix because it is the future label.
- All other March metrics: excluded because they are future information and would cause leakage.
- `client_hash_id` and `content_hash_id`: kept only for grouping and joining, not used as model features.
- `report_date` and `month`: used to define the February window, not used as predictive features.
- `client_has_gsc` and `client_has_ga4`: excluded because they are client-level availability flags, not content-performance measurements.
- `gsc_data_available` and `ga4_data_available`: used for safe filtering, not included directly in the five-feature vector.
- Raw client names, URLs, and private queries: not present or used.

Limitation: this slice measures directional relationships, not causation. Rows without usable March GA4 data are excluded from the labeled frame, so the result may not represent every client-content pair. The features also cannot explain external causes such as algorithm changes, seasonality, or content edits.

## Self-check

- The feature frame contains exactly five February features.
- March GA4 sessions is kept separate as the future label.
- Missing feature values are handled.
- The deliberate leakage experiment shows an artificially high score.
- The leaky column is removed from the final feature matrix.
- Client identifiers are used only for grouping and joining.
- No client names, URLs, tokens, or private queries are included.
- The notebook runs from top to bottom without errors.
- The notebook is committed under `work/notebooks/w03_feature_leakage_check.ipynb`.